# 00 · Data Ingest
Load Citi Bike July 2025 trips into DuckDB and run sanity checks.

## Setup
Check DuckDB version.

In [1]:
import duckdb
duckdb.__version__

'1.5.6'

## Load raw trips
Read all CSVs into a DuckDB table; inspect schema.

In [2]:
con = duckdb.connect("../data/citibike.duckdb")
con.sql("SET memory_limit = '4GB'") 

con.sql("""
CREATE OR REPLACE TABLE trips_raw AS
SELECT *
FROM read_csv('../data/raw/**/*.csv',
              union_by_name = true,
              types = {'start_station_id': 'VARCHAR',
                       'end_station_id':   'VARCHAR'})
""")

con.sql("DESCRIBE trips_raw").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ ride_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rideable_type      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ started_at         │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ ended_at           │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ start_station_name │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_station_id   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ end_station_name   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ end_station_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_lat          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │

## Row count & date range

In [3]:
con.sql("""
SELECT COUNT(*) AS n_trips, MIN(started_at) AS first_trip, MAX(started_at) AS last_trip
FROM trips_raw
""").show()

┌─────────┬─────────────────────────┬─────────────────────────┐
│ n_trips │       first_trip        │        last_trip        │
│  int64  │        timestamp        │        timestamp        │
├─────────┼─────────────────────────┼─────────────────────────┤
│ 4988053 │ 2025-06-29 23:16:34.475 │ 2025-07-31 23:57:27.342 │
└─────────┴─────────────────────────┴─────────────────────────┘



## Missing stations
Trips with no start / end station.

In [4]:
con.sql("""
SELECT
  COUNT(*) FILTER (WHERE start_station_id IS NULL) AS null_start,
  COUNT(*) FILTER (WHERE end_station_id   IS NULL) AS null_end
FROM trips_raw
""").show()

┌────────────┬──────────┐
│ null_start │ null_end │
│   int64    │  int64   │
├────────────┼──────────┤
│       2997 │    20399 │
└────────────┴──────────┘



## Bike type × rider type

In [5]:
con.sql("""
SELECT rideable_type, member_casual, COUNT(*) AS n
FROM trips_raw GROUP BY ALL ORDER BY n DESC
""").show()

┌───────────────┬───────────────┬─────────┐
│ rideable_type │ member_casual │    n    │
│    varchar    │    varchar    │  int64  │
├───────────────┼───────────────┼─────────┤
│ electric_bike │ member        │ 2738273 │
│ classic_bike  │ member        │ 1194479 │
│ electric_bike │ casual        │  792538 │
│ classic_bike  │ casual        │  262763 │
└───────────────┴───────────────┴─────────┘



## Trip duration (minutes)
1st / 50th / 99th percentiles.

In [6]:
con.sql("""
SELECT quantile_cont(date_diff('minute', started_at, ended_at), [0.01, 0.5, 0.99]) AS duration_q
FROM trips_raw
""").show()

┌───────────────────┐
│    duration_q     │
│     double[]      │
├───────────────────┤
│ [1.0, 10.0, 66.0] │
└───────────────────┘



## Station count

In [7]:
con.sql("SELECT COUNT(DISTINCT start_station_id) AS n_stations FROM trips_raw").show()

┌────────────┐
│ n_stations │
│   int64    │
├────────────┤
│       2157 │
└────────────┘



## Clean trips
Keep July starts, 1 min – 3 hr trips.

In [9]:
con.sql("""
CREATE OR REPLACE TABLE trips_clean AS
SELECT *,
       date_diff('second', started_at, ended_at) / 60.0 AS duration_min
FROM trips_raw
WHERE started_at >= '2025-07-01' AND started_at < '2025-08-01'
  AND date_diff('second', started_at, ended_at) BETWEEN 60 AND 3 * 3600
""")

con.sql("""
WITH counts AS (
  SELECT (SELECT COUNT(*) FROM trips_raw)   AS n_raw,
         (SELECT COUNT(*) FROM trips_clean) AS n_clean
)
SELECT n_raw, n_clean, 1 - n_clean / n_raw AS pct_removed
FROM counts
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬─────────┬───────────────────────┐
│  n_raw  │ n_clean │      pct_removed      │
│  int64  │  int64  │        double         │
├─────────┼─────────┼───────────────────────┤
│ 4988053 │ 4977167 │ 0.0021824146615924134 │
└─────────┴─────────┴───────────────────────┘



## Station metadata (GBFS)
Capacity + location; join on short_name.

In [10]:
con.sql("""
CREATE OR REPLACE TABLE stations AS
SELECT s.short_name AS station_id, s.name, s.lat, s.lon AS lng, s.capacity
FROM (
  SELECT unnest(data.stations) AS s
  FROM read_json('../data/external/station_information_20261101.json')
)
""")

# 有多少 July trips 對得到 GBFS station
con.sql("""
SELECT COUNT(*) FILTER (WHERE s.station_id IS NOT NULL) / COUNT(*) AS match_rate
FROM trips_clean t
LEFT JOIN stations s ON t.start_station_id = s.station_id
""").show()

┌────────────────────┐
│     match_rate     │
│       double       │
├────────────────────┤
│ 0.9641607364189307 │
└────────────────────┘



## Choose study area
Compare candidate bounding boxes.

In [11]:
con.sql("""
CREATE OR REPLACE TABLE station_loc AS
SELECT start_station_id      AS station_id,
       any_value(start_station_name) AS name,
       median(start_lat)     AS lat,
       median(start_lng)     AS lng,
       COUNT(*)              AS n_departures
FROM trips_clean
WHERE start_station_id IS NOT NULL
GROUP BY 1
""")

boxes = {
    "midtown (34th–59th)":     (40.745, 40.770, -74.010, -73.965),
    "midtown + 14th–34th":     (40.730, 40.770, -74.012, -73.965),
}

for name, (lat_min, lat_max, lng_min, lng_max) in boxes.items():
    print(name)
    con.sql(f"""
    SELECT COUNT(*) AS n_stations, SUM(n_departures) AS n_departures
    FROM station_loc
    WHERE lat BETWEEN {lat_min} AND {lat_max}
      AND lng BETWEEN {lng_min} AND {lng_max}
    """).show()

midtown (34th–59th)
┌────────────┬──────────────┐
│ n_stations │ n_departures │
│   int64    │    int128    │
├────────────┼──────────────┤
│        135 │       871700 │
└────────────┴──────────────┘

midtown + 14th–34th
┌────────────┬──────────────┐
│ n_stations │ n_departures │
│   int64    │    int128    │
├────────────┼──────────────┤
│        227 │      1518949 │
└────────────┴──────────────┘



## Study area: 14th–59th St
Fix station list; check capacity coverage and AM net flow.

In [12]:
con.sql("""
CREATE OR REPLACE TABLE study_stations AS
SELECT l.station_id, l.name, l.lat, l.lng, l.n_departures, s.capacity
FROM station_loc l
LEFT JOIN stations s USING (station_id)
WHERE l.lat BETWEEN 40.730 AND 40.770
  AND l.lng BETWEEN -74.012 AND -73.965
""")

# capacity 覆蓋率
con.sql("""
SELECT COUNT(*)                                   AS n_stations,
       COUNT(*) FILTER (WHERE capacity IS NULL)   AS missing_capacity,
       SUM(n_departures) FILTER (WHERE capacity IS NULL) AS departures_at_missing
FROM study_stations
""").show()

# 平日早上 7–10 點各站 net flow（arrivals − departures）
con.sql("""
WITH am AS (
  SELECT start_station_id AS station_id, -1 AS flow
  FROM trips_clean
  WHERE hour(started_at) BETWEEN 7 AND 9 AND dayofweek(started_at) BETWEEN 1 AND 5
  UNION ALL
  SELECT end_station_id, 1
  FROM trips_clean
  WHERE hour(ended_at) BETWEEN 7 AND 9 AND dayofweek(ended_at) BETWEEN 1 AND 5
    AND end_station_id IS NOT NULL
),
net AS (
  SELECT station_id, SUM(flow) / 23.0 AS avg_daily_net   -- 2025 年 7 月有 23 個平日
  FROM am
  WHERE station_id IN (SELECT station_id FROM study_stations)
  GROUP BY 1
)
SELECT COUNT(*) FILTER (WHERE avg_daily_net < -5) AS outflow_stations,
       COUNT(*) FILTER (WHERE avg_daily_net >  5) AS inflow_stations,
       COUNT(*) FILTER (WHERE avg_daily_net BETWEEN -5 AND 5) AS balanced_stations,
       ROUND(SUM(avg_daily_net), 1) AS area_total_net
FROM net
""").show()

┌────────────┬──────────────────┬───────────────────────┐
│ n_stations │ missing_capacity │ departures_at_missing │
│   int64    │      int64       │        int128         │
├────────────┼──────────────────┼───────────────────────┤
│        227 │               17 │                 83261 │
└────────────┴──────────────────┴───────────────────────┘

┌──────────────────┬─────────────────┬───────────────────┬────────────────┐
│ outflow_stations │ inflow_stations │ balanced_stations │ area_total_net │
│      int64       │      int64      │       int64       │     double     │
├──────────────────┼─────────────────┼───────────────────┼────────────────┤
│               75 │             110 │                42 │         2198.5 │
└──────────────────┴─────────────────┴───────────────────┴────────────────┘



## Fill missing capacity
Name match first, then area median.

In [13]:
con.sql("""
CREATE OR REPLACE TABLE study_stations AS
WITH name_match AS (
  SELECT name, any_value(capacity) AS cap_by_name
  FROM stations GROUP BY 1
),
med AS (
  SELECT median(capacity) AS cap_median FROM study_stations WHERE capacity IS NOT NULL
)
SELECT s.station_id, s.name, s.lat, s.lng, s.n_departures,
       COALESCE(s.capacity, n.cap_by_name, m.cap_median)::INT AS capacity,
       CASE WHEN s.capacity IS NOT NULL  THEN 'gbfs_id'
            WHEN n.cap_by_name IS NOT NULL THEN 'gbfs_name'
            ELSE 'imputed_median' END AS capacity_source
FROM study_stations s
LEFT JOIN name_match n USING (name)
CROSS JOIN med m
""")

con.sql("""
SELECT capacity_source, COUNT(*) AS n_stations, SUM(n_departures) AS n_departures
FROM study_stations GROUP BY 1 ORDER BY 2 DESC
""").show()

┌─────────────────┬────────────┬──────────────┐
│ capacity_source │ n_stations │ n_departures │
│     varchar     │   int64    │    int128    │
├─────────────────┼────────────┼──────────────┤
│ gbfs_id         │        210 │      1435688 │
│ imputed_median  │         14 │        78982 │
│ gbfs_name       │          3 │         4279 │
└─────────────────┴────────────┴──────────────┘



## Daily net flow cycle
Weekday hourly net flow for the whole area.

In [15]:
con.sql("""
WITH flows AS (
  SELECT hour(started_at) AS hr, -1 AS flow
  FROM trips_clean
  WHERE dayofweek(started_at) BETWEEN 1 AND 5
    AND start_station_id IN (SELECT station_id FROM study_stations)
  UNION ALL
  SELECT hour(ended_at), 1
  FROM trips_clean
  WHERE dayofweek(ended_at) BETWEEN 1 AND 5
    AND end_station_id IN (SELECT station_id FROM study_stations)
)
SELECT hr,
       ROUND(SUM(flow) / 23.0, 1) AS avg_net,
       ROUND(SUM(SUM(flow)) OVER (ORDER BY hr) / 23.0, 1) AS cum_net
FROM flows
GROUP BY hr ORDER BY hr
""").show(max_rows=24)

┌───────┬─────────┬─────────┐
│  hr   │ avg_net │ cum_net │
│ int64 │ double  │ double  │
├───────┼─────────┼─────────┤
│     0 │   -32.5 │   -32.5 │
│     1 │   -19.9 │   -52.4 │
│     2 │   -10.7 │   -63.1 │
│     3 │     3.2 │   -60.0 │
│     4 │    -3.8 │   -63.7 │
│     5 │   -24.9 │   -88.6 │
│     6 │    38.2 │   -50.4 │
│     7 │   327.7 │   277.3 │
│     8 │   887.5 │  1164.8 │
│     9 │   983.3 │  2148.1 │
│    10 │   276.0 │  2424.1 │
│    11 │    33.7 │  2457.8 │
│    12 │    10.6 │  2468.3 │
│    13 │   -56.7 │  2411.6 │
│    14 │  -111.1 │  2300.5 │
│    15 │  -173.6 │  2126.9 │
│    16 │  -609.5 │  1517.4 │
│    17 │ -1124.3 │   393.2 │
│    18 │  -534.7 │  -141.5 │
│    19 │  -206.7 │  -348.2 │
│    20 │  -102.7 │  -450.9 │
│    21 │  -106.3 │  -557.2 │
│    22 │   -74.3 │  -631.5 │
│    23 │   -49.6 │  -681.0 │
└───────┴─────────┴─────────┘
  24 rows         3 columns



## Key findings
- July 2025: 4.98M trips; cleaning removed 0.22% (cross-month starts, <1 min, >3 hr).
- Study area (14th–59th St): 227 stations, 1.52M departures.
- Capacity: 210 from GBFS ID, 3 by name, 14 imputed with area median (5.2% of departures).
- Weekday commuter pattern: +2,450 net inflow by noon, strong outflow 4–7 PM.
- Area loses ~680 bikes per weekday → operator must resupply; model needs an external depot node.
- 12–6 AM is nearly idle → overnight rebalancing window.